In [25]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, hamming_loss, f1_score, precision_score, recall_score, classification_report

# Load training and validation datasets
train_df = pd.read_csv('train_data(1).csv')
val_df = pd.read_csv('val_data.csv')

# Handle missing values
train_df = train_df.dropna(subset=['book_details'])
val_df = val_df.dropna(subset=['book_details'])

# Memastikan bahwa label berada dalam format list
def convert_to_list(row):
    return [col for col in row.index if row[col] == 1]

train_df['genres'] = train_df[['Fantasy', 'Young Adult', 'Classics', 'Romance', 'Historical Fiction']].apply(convert_to_list, axis=1)
val_df['genres'] = val_df[['Fantasy', 'Young Adult', 'Classics', 'Romance', 'Historical Fiction']].apply(convert_to_list, axis=1)

# Feature and target
X_train = train_df['book_details']
y_train = train_df['genres'].tolist()
X_val = val_df['book_details']
y_val = val_df['genres'].tolist()

# TF-IDF vectorization
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)

# Binarize the labels
mlb = MultiLabelBinarizer()
y_train_bin = mlb.fit_transform(y_train)
y_val_bin = mlb.transform(y_val)

print("Classes:", mlb.classes_)



Classes: ['Classics' 'Fantasy' 'Historical Fiction' 'Romance' 'Young Adult']


In [26]:
# Build the SVM model
svm_model = Pipeline([
    ('clf', OneVsRestClassifier(SVC(kernel='linear')))
])

# Fit the model
svm_model.fit(X_train_tfidf, y_train_bin)

# Predict on the validation set
y_pred_bin = svm_model.predict(X_val_tfidf)

# Evaluate the model
accuracy = accuracy_score(y_val_bin, y_pred_bin)
hamming = hamming_loss(y_val_bin, y_pred_bin)
f1 = f1_score(y_val_bin, y_pred_bin, average='micro')
precision = precision_score(y_val_bin, y_pred_bin, average='micro')
recall = recall_score(y_val_bin, y_pred_bin, average='micro')

print(f"Validation Accuracy: {accuracy:.2f}")
print(f"Hamming Loss: {hamming:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")

report = classification_report(y_val_bin, y_pred_bin, target_names=mlb.classes_)
print(report)


Validation Accuracy: 0.48
Hamming Loss: 0.15
F1-Score: 0.69
Precision: 0.79
Recall: 0.61
                    precision    recall  f1-score   support

          Classics       0.80      0.57      0.66       405
           Fantasy       0.84      0.74      0.79       604
Historical Fiction       0.76      0.51      0.61       366
           Romance       0.76      0.53      0.62       402
       Young Adult       0.76      0.63      0.69       396

         micro avg       0.79      0.61      0.69      2173
         macro avg       0.78      0.60      0.67      2173
      weighted avg       0.79      0.61      0.69      2173
       samples avg       0.56      0.50      0.51      2173



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [29]:
# Convert predicted binary values back to original label format
y_pred_labels = mlb.inverse_transform(y_pred_bin)
y_val_labels = mlb.inverse_transform(y_val_bin)

# Display some samples of the predicted labels vs true labels
for i in range(5):
    print(f"Book details: {X_val.iloc[i]}")
    print(f"True labels: {y_val_labels[i]}")
    print(f"Predicted labels: {y_pred_labels[i]}")
    print('-' * 50)

accuracy = accuracy_score(y_val_bin, y_pred_bin)
hamming = hamming_loss(y_val_bin, y_pred_bin)
f1 = f1_score(y_val_bin, y_pred_bin, average='micro')  # 'micro' averages by considering each element separately
precision = precision_score(y_val_bin, y_pred_bin, average='micro')
recall = recall_score(y_val_bin, y_pred_bin, average='micro')


print(f"Validation Accuracy: {accuracy:.2f}")
print(f"Hamming Loss: {hamming:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")

Book details: hrs min h lawrence controversial classic rainbow follow life love three generation brangwen family tempestuous relationship play backdrop change witness arrival industrialization constant unende attempt grasp high form existence symbolize persistent unify motif rainbow lawrence fourth novel prequel woman love invigorate absorb tale undying determination human soul
True labels: ('Classics', 'Romance')
Predicted labels: ('Classics', 'Romance')
--------------------------------------------------
Book details: mackenzie allen philips young daughter missy abducted family vacation evidence may brutally murder find abandon shack deep oregon wilderness four year later midst great sadness mack receive suspicious note apparently god invite back shack weekend well judgment arrive shack wintry afternoon walk back darkest nightmare find change mack world forever world religion seems grow increasingly irrelevant shack wrestle timeless question god world fill unspeakable pain answer mack